# 09_Cross_Attention_Fusion.ipynb
Merge CNN and Transformer features and train a multimodal cross-attention classifier.

In [ ]:
!pip -q install torch pandas scikit-learn

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from google.colab import files

print("Upload cnn_features.csv")
u=files.upload()
cnn=pd.read_csv(list(u.keys())[0])

print("Upload transformer_features.csv")
u=files.upload()
tr=pd.read_csv(list(u.keys())[0])

cnn=cnn.rename(columns={c:f"cnn_{c}" for c in cnn.columns if c!="event_id"})
tr=tr.rename(columns={c:f"tr_{c}" for c in tr.columns if c!="event_id"})

df=cnn.merge(tr,on="event_id")

labels=[]
for e in df["event_id"].astype(str):
    if "FLASH" in e.upper():
        labels.append("FLASH")
    elif "HEAT" in e.upper():
        labels.append("HEAT")
    else:
        labels.append("OTHER")

le=LabelEncoder()
y=le.fit_transform(labels)

cnn_cols=[c for c in df.columns if c.startswith("cnn_")]
tr_cols=[c for c in df.columns if c.startswith("tr_")]

class FusionDataset(Dataset):
    def __init__(self,df,y):
        self.cnn=torch.tensor(df[cnn_cols].values,dtype=torch.float32)
        self.tr=torch.tensor(df[tr_cols].values,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return self.cnn[i],self.tr[i],self.y[i]

loader=DataLoader(FusionDataset(df,y),batch_size=8,shuffle=True)

class CrossAttentionFusion(nn.Module):
    def __init__(self,dim=512,classes=2):
        super().__init__()
        self.attn=nn.MultiheadAttention(embed_dim=dim,num_heads=8,batch_first=True)
        self.cls=nn.Sequential(
            nn.Linear(dim,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,classes)
        )
    def forward(self,cnn,tr):
        q=cnn.unsqueeze(1)
        k=tr.unsqueeze(1)
        v=tr.unsqueeze(1)
        fused,_=self.attn(q,k,v)
        fused=fused.squeeze(1)
        out=self.cls(fused)
        return out

device="cuda" if torch.cuda.is_available() else "cpu"
model=CrossAttentionFusion(classes=len(le.classes_)).to(device)
opt=torch.optim.Adam(model.parameters(),lr=1e-4)
loss_fn=nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    correct=0
    total=0
    for c,t,l in loader:
        c,t,l=c.to(device),t.to(device),l.to(device)
        opt.zero_grad()
        pred=model(c,t)
        loss=loss_fn(pred,l)
        loss.backward()
        opt.step()
        correct+=(pred.argmax(1)==l).sum().item()
        total+=l.size(0)
    print(f"Epoch {epoch+1}: Accuracy {100*correct/total:.2f}%")

torch.save(model.state_dict(),"multimodal_model.pth")
files.download("multimodal_model.pth")
print("Training complete.")
